# 03 - Quality, NFE, system speed 분해

**학습 목표**: 논문에 보고된 MMMU-Pro-V progressive rows를 작은 표로 검산하고 Pareto frontier를 구합니다. 논문 수치를 재계산할 뿐 모델 inference를 재현하지 않습니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
rows = [
    {'name': 'AR baseline', 'quality': 26.3, 'tps': 56.7},
    {'name': 'MDM tau=0.9', 'quality': 21.4, 'tps': 82.2},
    {'name': '+ linear self-spec', 'quality': 24.6, 'tps': 112.7},
    {'name': '+ SGLang', 'quality': 24.1, 'tps': 319.0},
    {'name': '+ FP8', 'quality': 23.8, 'tps': 350.3},
]
# 모든 단계의 TPS를 같은 AR baseline으로 나눠 누적 speedup을 검산합니다.
base = rows[0]['tps']
for row in rows:
    row['speedup'] = row['tps'] / base
    print(f"{row['name']:20s} quality={row['quality']:4.1f} TPS={row['tps']:5.1f} speedup={row['speedup']:0.2f}x")
assert round(rows[-1]['speedup'], 2) == 6.18

In [ ]:
def pareto_frontier(points):
    frontier = []
    for candidate in points:
        dominated = any(
            other['quality'] >= candidate['quality'] and other['tps'] >= candidate['tps']
            and (other['quality'] > candidate['quality'] or other['tps'] > candidate['tps'])
            for other in points if other is not candidate
        )
        if not dominated:
            frontier.append(candidate['name'])
    return frontier

print('Pareto frontier:', pareto_frontier(rows))
threshold_rows = [(1.0, 21.6, 1.00), (0.9, 21.4, 1.95), (0.4, 18.5, 2.90)]
for tau, quality, tokens_per_step in threshold_rows:
    print(f'tau={tau:0.1f}: quality={quality:0.1f}, tokens/step={tokens_per_step:0.2f}')

6.18x는 diffusion algorithm, speculative verification, serving engine, quantization을 누적한 수치이며 AR과 품질이 완전히 같지 않습니다. Short-answer 평균 동률(74.0)과 이 long-answer/FP8 row를 섞어 말하지 않아야 합니다.